# Stage 10 — Preassigned Development Extension and Locked-Blind Dataset Registry

This notebook freezes **which new datasets may be used for development and which are permanently reserved for the final blind test**. It is a governance and acquisition-planning stage, not a data-acquisition, model-fitting, transfer-evaluation, or blind-unsealing stage.

It verifies the immutable Stage 9 specification; freezes harmonised endpoints, label mappings, patient-grouping requirements, licences, official source records, acquisition order, candidate development edges, locked-blind target edges, overlap gates, and label-quarantine rules; then writes a Stage 11 handoff that contains development assets only.

No image archive, diagnostic label, embedding, new source model, target outcome, fitted DDO2 coefficient, or blind prediction is read or produced here. A dataset that becomes unavailable later remains unavailable under its frozen role. It cannot be moved between development and blind, and it cannot be replaced ad hoc after outcomes are seen.


In [8]:
# @title 10-0. Mount Drive, verify Stage 9, and establish a no-data-access boundary
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
except ModuleNotFoundError:
    DEFAULT_PROJECT_ROOT = Path("/tmp/Cross-Modal_Diagnostic_Observability")

print("================ STAGE 10 ROLE-REGISTRY PREFLIGHT ================")

PROJECT_ROOT = Path(os.environ.get("CDO_PROJECT_ROOT", str(DEFAULT_PROJECT_ROOT)))
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
CROSS_MODAL_ROOT = PROJECT_ROOT / "06_Data_Records" / "Cross_Modal"
STAGE9_ROOT = CROSS_MODAL_ROOT / "Stage9_Hierarchical_DDO2_Specification_Freeze_v0.1"
STAGE10_ROOT = CROSS_MODAL_ROOT / "Stage10_Preassigned_Development_And_Locked_Blind_Registry_v0.1"

PROTOCOL_ROOT = STAGE10_ROOT / "00_Protocol"
REGISTRY_ROOT = STAGE10_ROOT / "01_Role_Registry"
AUDIT_ROOT = STAGE10_ROOT / "02_Endpoint_And_Access_Audit"
DEVELOPMENT_ROOT = STAGE10_ROOT / "03_Development_Extension"
BLIND_ROOT = STAGE10_ROOT / "04_Locked_Blind_Reserve"
RESULT_ROOT = STAGE10_ROOT / "05_Results"
for directory in [CODE_ROOT, PROTOCOL_ROOT, REGISTRY_ROOT, AUDIT_ROOT, DEVELOPMENT_ROOT, BLIND_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = CODE_ROOT / "CrossModal_Stage10_Preassigned_Development_And_Locked_Blind_Registry_v0.1.ipynb"
STAGE9_PROTOCOL_ROOT = STAGE9_ROOT / "00_Protocol"
STAGE9_SPEC_ROOT = STAGE9_ROOT / "02_Frozen_Specification"
STAGE9_BLIND_ROOT = STAGE9_ROOT / "04_Future_Blind_Validation"
STAGE9_RESULT_ROOT = STAGE9_ROOT / "05_Results"

STAGE9_FINAL_PATH = STAGE9_RESULT_ROOT / "Stage9_Hierarchical_DDO2_Specification_Freeze_Complete_v0.1.json"
STAGE9_SEAL_PATH = STAGE9_PROTOCOL_ROOT / "Stage9_Hierarchical_DDO2_Specification_Protocol_Seal_v0.1.json"
STAGE9_MODEL_PATH = STAGE9_SPEC_ROOT / "Stage9_Frozen_Hierarchical_DDO2_Model_Specification_v0.1.json"
STAGE9_ABSTAIN_PATH = STAGE9_SPEC_ROOT / "Stage9_Frozen_DDO2_ABSTAIN_Policy_v0.1.json"
STAGE9_FUTURE_BLIND_PATH = STAGE9_BLIND_ROOT / "Stage9_Frozen_Future_Blind_Validation_Protocol_v0.1.json"
STAGE9_ROLE_TEMPLATE_PATH = STAGE9_BLIND_ROOT / "Stage9_Stage10_Acquisition_Role_Registry_Template_v0.1.csv"
STAGE9_AXIS_SHORTFALL_PATH = STAGE9_BLIND_ROOT / "Stage9_Axis_Class_Balance_Shortfall_v0.1.csv"
STAGE9_REQUIREMENTS_PATH = STAGE9_BLIND_ROOT / "Stage9_Next_Stage_Data_Requirement_Summary_v0.1.csv"

PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage10_Acquisition_Role_Registry_Protocol_Seal_v0.1.json"
INPUT_COMMITMENT_PATH = PROTOCOL_ROOT / "Stage10_Parent_Input_Commitment_v0.1.csv"
ENDPOINT_SPEC_PATH = REGISTRY_ROOT / "Stage10_Frozen_Harmonised_Endpoint_Specifications_v0.1.json"
ROLE_REGISTRY_PATH = REGISTRY_ROOT / "Stage10_Frozen_Dataset_Acquisition_Role_Registry_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage10_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Stage10_Preassigned_Role_Registry_Complete_v0.1.json"
OUTPUT_MANIFEST_PATH = RESULT_ROOT / "Stage10_Output_Integrity_Manifest_v0.1.csv"
REPORT_PATH = RESULT_ROOT / "Stage10_Preassigned_Role_Registry_Report_v0.1.md"
FIGURE_PATH = RESULT_ROOT / "Stage10_Development_And_Blind_Capacity_v0.1.png"

EXPECTED_STAGE9_FINAL_HASH = "2568a9fdaff83ab74938655d65d25cd9d50f8b28bafbc44e856098932f6c6648"
EXPECTED_STAGE9_PROTOCOL_SEAL = "f5689995ec6ab0dcf0bcacf813d578edd47edbedf8fb96b7809ec674e3cad593"
EXPECTED_MODEL_SPEC_HASH = "e2a0d8f65143e0b9d58d740595a39b803f1684481c4f8af01f05935bfaacb732"
EXPECTED_ABSTAIN_POLICY_HASH = "a7d744dfdec1b4f0847d2e6b22a9e6368738630d3b96b3328584ce0ebf1da9b0"
EXPECTED_FUTURE_BLIND_PROTOCOL_HASH = "acef1a8da90b95e875ed29572ca4a8283a5035ebfc1a1bf9fc787c487a01ec5a"
MAXIMUM_NEW_STAGE10_BYTES = 32 * 1024 * 1024

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def sha256_json(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_csv_text(frame):
    return frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")

def write_immutable_text(path, text):
    path = Path(path)
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing immutable text differs: {path}"
    else:
        path.write_text(text, encoding="utf-8")

def write_immutable_csv(path, frame):
    write_immutable_text(path, canonical_csv_text(frame))

def write_immutable_json(path, payload):
    write_immutable_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def atomic_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary, path)

def verify_self_hashed_json(path, hash_field, expected_hash):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    claim = payload.get(hash_field)
    assert claim == expected_hash, f"Unexpected {hash_field}: {path}"
    without_claim = dict(payload)
    without_claim.pop(hash_field)
    assert sha256_json(without_claim) == claim, f"Self-hash mismatch: {path}"
    return payload

def normalised_notebook_source_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else str(source)
        payload.append({"cell_type": cell["cell_type"], "source": source.replace("\r\n", "\n")})
    return sha256_json(payload)

required_parent_inputs = [
    NOTEBOOK_PATH, STAGE9_FINAL_PATH, STAGE9_SEAL_PATH, STAGE9_MODEL_PATH, STAGE9_ABSTAIN_PATH,
    STAGE9_FUTURE_BLIND_PATH, STAGE9_ROLE_TEMPLATE_PATH, STAGE9_AXIS_SHORTFALL_PATH, STAGE9_REQUIREMENTS_PATH,
]
assert all(path.is_file() for path in required_parent_inputs), "A frozen Stage 9 input is missing; do not substitute a similarly named file."

stage9_final = verify_self_hashed_json(STAGE9_FINAL_PATH, "final_record_sha256", EXPECTED_STAGE9_FINAL_HASH)
stage9_seal = verify_self_hashed_json(STAGE9_SEAL_PATH, "seal_sha256", EXPECTED_STAGE9_PROTOCOL_SEAL)
stage9_model = verify_self_hashed_json(STAGE9_MODEL_PATH, "specification_sha256", EXPECTED_MODEL_SPEC_HASH)
stage9_abstain = verify_self_hashed_json(STAGE9_ABSTAIN_PATH, "policy_sha256", EXPECTED_ABSTAIN_POLICY_HASH)
stage9_future_blind = verify_self_hashed_json(STAGE9_FUTURE_BLIND_PATH, "protocol_sha256", EXPECTED_FUTURE_BLIND_PROTOCOL_HASH)

assert stage9_final["stage9_protocol_seal_sha256"] == EXPECTED_STAGE9_PROTOCOL_SEAL
assert stage9_final["model_specification_sha256"] == EXPECTED_MODEL_SPEC_HASH
assert stage9_final["abstain_policy_sha256"] == EXPECTED_ABSTAIN_POLICY_HASH
assert stage9_final["future_blind_protocol_sha256"] == EXPECTED_FUTURE_BLIND_PROTOCOL_HASH
assert stage9_final["fit_authorised"] is False and stage9_final["future_blind_labels_accessed"] is False

parent_paths = required_parent_inputs[1:]
parent_commitment = pd.DataFrame([
    {
        "role": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in parent_paths
])
write_immutable_csv(INPUT_COMMITMENT_PATH, parent_commitment)

notebook_source_hash = normalised_notebook_source_sha256(NOTEBOOK_PATH)
runtime_state = {
    "stage": "Stage10", "parent_stage9_verified": True, "role_registry_sealed": False,
    "external_requests_made": False, "image_archives_downloaded": False, "future_labels_accessed": False,
    "ddo2_coefficients_fitted": False, "blind_validation_performed": False, "last_updated_utc": utc_now(),
}
atomic_json(RUNTIME_STATE_PATH, runtime_state)

assert not (STAGE10_ROOT / "raw_data").exists(), "Raw data are prohibited in Stage 10."
print("Stage 9 final record verified:", EXPECTED_STAGE9_FINAL_HASH)
print("Stage 9 protocol / model / ABSTAIN / future blind hashes verified.")
print("External requests / image downloads / future label access:", False, "/", False, "/", False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
================ STAGE 10 ROLE-REGISTRY PREFLIGHT ================
Stage 9 final record verified: 2568a9fdaff83ab74938655d65d25cd9d50f8b28bafbc44e856098932f6c6648
Stage 9 protocol / model / ABSTAIN / future blind hashes verified.
External requests / image downloads / future label access: False / False / False


In [9]:
# @title 10-1. Freeze harmonised endpoints and mutually exclusive dataset roles
ENDPOINT_SPECS = {
    "breast_ultrasound": {
        "task": "breast_lesion_malignant_vs_benign",
        "positive_class": "released definitive malignant pathology or released malignant binary diagnosis",
        "negative_class": "released definitive benign pathology or released benign binary diagnosis",
        "excluded": ["normal breast without a lesion", "unknown", "indeterminate", "missing diagnosis"],
        "forbidden_surrogate": "BI-RADS category alone may not be converted into pathology",
        "prediction_unit": "released B-mode image or deterministically rendered scan",
        "grouping_unit": "patient; lesion nested within patient when lesion identifiers exist",
        "split_rule": "all images or scans from one patient remain in one split",
        "metric_rule": "patient-clustered confidence intervals; no image-row bootstrap",
        "multi_image_rule": "retain released images; group all splitting and resampling by patient",
    },
    "dermoscopy": {
        "task": "melanoma_vs_melanocytic_nevus",
        "positive_class": "melanoma",
        "negative_class": "melanocytic nevus",
        "excluded": ["basal cell carcinoma", "seborrhoeic keratosis", "miscellaneous and non-melanocytic diagnoses"],
        "forbidden_surrogate": "clinical-image or checklist score may not replace the released diagnosis",
        "prediction_unit": "dermoscopic image only; clinical photographs are excluded",
        "grouping_unit": "lesion or patient where available",
        "split_rule": "all images from one lesion or patient remain in one split",
        "metric_rule": "lesion/patient-clustered confidence intervals",
        "multi_image_rule": "average logits within a lesion before endpoint evaluation if multiple dermoscopic views exist",
    },
}
write_immutable_json(ENDPOINT_SPEC_PATH, ENDPOINT_SPECS)
ENDPOINT_SPEC_HASH = sha256_file(ENDPOINT_SPEC_PATH)

DEVELOPMENT = "DEVELOPMENT_EXTENSION"
BLIND = "LOCKED_BLIND_TEST"
SOURCE_DATE = "2026-07-21"

dataset_rows = [
    {
        "dataset_id": "BUS_BRA_2024", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": DEVELOPMENT, "acquisition_priority": 1,
        "analysis_use": "SOURCE_OR_TARGET_AFTER_PATIENT_AND_SOURCE_GATES", "labels_permitted_before_label_free_prediction_freeze": True,
        "eligible_for_model_or_scaler_fit": True, "eligible_for_final_blind_claim": False,
        "source_candidacy": "ELIGIBLE_IF_PATIENT_GROUPING_AND_SOURCE_RECOVERABILITY_PASS",
        "official_landing_url": "https://zenodo.org/records/8231412", "persistent_id": "10.5281/zenodo.8231412",
        "repository_version": "Zenodo record 8231412 version 1.0", "license": "CC BY 4.0",
        "expected_images_or_scans": 1875, "expected_patients": 1064, "patient_grouping_status": "RELEASED_PATIENT_IDENTIFIERS_EXPECTED_VERIFY_AT_ACQUISITION",
        "positive_mapping": "released pathology=malignant", "negative_mapping": "released pathology=benign",
        "exclusions": "normal|unknown|missing; never infer pathology from BI-RADS",
        "label_storage_coupling": "UNKNOWN_UNTIL_ACQUISITION", "acquisition_gate": "OFFICIAL_ARCHIVE_HASH_AND_SCHEMA_CHECK",
        "provenance_overlap_gate": "NO_KNOWN_OVERLAP_NOT_YET_HASH_PROVEN", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "BUSI_WHU_2025_V3", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": DEVELOPMENT, "acquisition_priority": 2,
        "analysis_use": "TARGET_ONLY_UNTIL_PATIENT_GROUPING_PROVEN_THEN_SOURCE_CANDIDATE", "labels_permitted_before_label_free_prediction_freeze": True,
        "eligible_for_model_or_scaler_fit": True, "eligible_for_final_blind_claim": False,
        "source_candidacy": "HOLD_AS_SOURCE_UNTIL_PATIENT_IDENTIFIERS_AND_SOURCE_GATE_PASS",
        "official_landing_url": "https://data.mendeley.com/datasets/k6cpmwybk3/3", "persistent_id": "10.17632/k6cpmwybk3.3",
        "repository_version": "Mendeley Data version 3", "license": "CC BY 4.0",
        "expected_images_or_scans": 927, "expected_patients": -1, "patient_grouping_status": "VERIFY_BEFORE_SOURCE_USE",
        "positive_mapping": "released diagnosis=malignant", "negative_mapping": "released diagnosis=benign",
        "exclusions": "unknown|missing|non-binary", "label_storage_coupling": "UNKNOWN_UNTIL_ACQUISITION",
        "acquisition_gate": "OFFICIAL_ARCHIVE_HASH_SCHEMA_AND_PATIENT_GROUP_CHECK",
        "provenance_overlap_gate": "NO_KNOWN_OVERLAP_NOT_YET_HASH_PROVEN", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "BREAST_LESIONS_USG_2024", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": DEVELOPMENT, "acquisition_priority": 3,
        "analysis_use": "SOURCE_OR_TARGET_AFTER_PATIENT_AND_SOURCE_GATES", "labels_permitted_before_label_free_prediction_freeze": True,
        "eligible_for_model_or_scaler_fit": True, "eligible_for_final_blind_claim": False,
        "source_candidacy": "ELIGIBLE_IF_RELEASED_DIAGNOSIS_AND_SOURCE_RECOVERABILITY_PASS",
        "official_landing_url": "https://www.cancerimagingarchive.net/collection/breast-lesions-usg/", "persistent_id": "10.7937/9WKK-Q141",
        "repository_version": "TCIA collection current at source-evidence date", "license": "CC BY 4.0",
        "expected_images_or_scans": 256, "expected_patients": 256, "patient_grouping_status": "ONE_RELEASED_SCAN_PER_PATIENT_EXPECTED_VERIFY_AT_ACQUISITION",
        "positive_mapping": "released confirmed diagnosis=malignant", "negative_mapping": "released confirmed diagnosis=benign",
        "exclusions": "normal|unknown|missing; never infer pathology from BI-RADS",
        "label_storage_coupling": "SUPPORTING_TABLE_EXPECTED_SEPARATE_FROM_IMAGES", "acquisition_gate": "TCIA_VERSION_MANIFEST_AND_LABEL_SCHEMA_CHECK",
        "provenance_overlap_gate": "REQUIRE_CLEARANCE_AGAINST_OASBUD_BEFORE_ANY_BLIND_EDGE", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "BUS_UCLM_2025_V3", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": DEVELOPMENT, "acquisition_priority": 4,
        "analysis_use": "SOURCE_OR_TARGET_AFTER_PATIENT_AND_SOURCE_GATES", "labels_permitted_before_label_free_prediction_freeze": True,
        "eligible_for_model_or_scaler_fit": True, "eligible_for_final_blind_claim": False,
        "source_candidacy": "ELIGIBLE_IF_PATIENT_GROUPING_AND_SOURCE_RECOVERABILITY_PASS",
        "official_landing_url": "https://data.mendeley.com/datasets/7fvgj4jsp7/3", "persistent_id": "10.17632/7fvgj4jsp7.3",
        "repository_version": "Mendeley Data version 3", "license": "CC BY-NC 3.0",
        "expected_images_or_scans": 683, "expected_patients": 38, "patient_grouping_status": "RELEASED_PATIENT_GROUPS_EXPECTED_VERIFY_AT_ACQUISITION",
        "positive_mapping": "released lesion/mask class=malignant", "negative_mapping": "released lesion/mask class=benign",
        "exclusions": "normal|unknown|missing", "label_storage_coupling": "MASK_COLORS_ENCODE_CLASS",
        "acquisition_gate": "PATIENT_GROUP_AND_MASK_TO_DIAGNOSIS_MAPPING_CHECK",
        "provenance_overlap_gate": "NO_KNOWN_OVERLAP_NOT_YET_HASH_PROVEN", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "RODRIGUES_BUI_2017", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": DEVELOPMENT, "acquisition_priority": 5,
        "analysis_use": "TARGET_ONLY_UNTIL_PATIENT_GROUPING_PROVEN_THEN_SOURCE_CANDIDATE", "labels_permitted_before_label_free_prediction_freeze": True,
        "eligible_for_model_or_scaler_fit": True, "eligible_for_final_blind_claim": False,
        "source_candidacy": "HOLD_AS_SOURCE_UNTIL_PATIENT_IDENTIFIERS_DEDUPLICATION_AND_SOURCE_GATE_PASS",
        "official_landing_url": "https://data.mendeley.com/datasets/wmy84gzngw/1", "persistent_id": "10.17632/wmy84gzngw.1",
        "repository_version": "Mendeley Data version 1", "license": "CC BY 4.0",
        "expected_images_or_scans": 250, "expected_patients": -1, "patient_grouping_status": "NOT_ESTABLISHED_BEFORE_ACQUISITION",
        "positive_mapping": "released class=malignant", "negative_mapping": "released class=benign",
        "exclusions": "unknown|missing|derived copies", "label_storage_coupling": "UNKNOWN_UNTIL_ACQUISITION",
        "acquisition_gate": "PATIENT_GROUP_DEDUPLICATION_AND_DERIVED_COPY_CHECK",
        "provenance_overlap_gate": "REQUIRE_HASH_CLEARANCE_AGAINST_ALL_ULTRASOUND_DATASETS", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "BUSI_CAIRO_2019", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": BLIND, "acquisition_priority": 101,
        "analysis_use": "LOCKED_TARGET_ONLY_NEVER_SOURCE", "labels_permitted_before_label_free_prediction_freeze": False,
        "eligible_for_model_or_scaler_fit": False, "eligible_for_final_blind_claim": True, "source_candidacy": "PROHIBITED_BLIND_TARGET_ONLY",
        "official_landing_url": "https://scholar.cu.edu.eg/?q=afahmy/pages/dataset", "persistent_id": "10.1016/j.dib.2019.104863",
        "repository_version": "authors' released BUSI dataset", "license": "CC BY 4.0 per data article",
        "expected_images_or_scans": 780, "expected_patients": 600, "patient_grouping_status": "RELEASED_COHORT_COUNT_KNOWN_GROUP_MAP_MUST_BE_VERIFIED_WITHOUT_OUTCOME_USE",
        "positive_mapping": "sealed released class=malignant", "negative_mapping": "sealed released class=benign",
        "exclusions": "normal|unknown|missing", "label_storage_coupling": "CLASS_NAMES_COUPLED_TO_ARCHIVE_PATHS_OR_FILENAMES",
        "acquisition_gate": "POST_MODEL_FREEZE_CONTROLLED_OPAQUE_ID_EXTRACTION_OR_MARK_UNAVAILABLE",
        "provenance_overlap_gate": "NO_KNOWN_OVERLAP_NOT_YET_HASH_PROVEN", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "OASBUD_2017", "modality": "breast_ultrasound", "task": "breast_lesion_malignant_vs_benign",
        "role_frozen_before_label_access": BLIND, "acquisition_priority": 102,
        "analysis_use": "LOCKED_TARGET_ONLY_NEVER_SOURCE", "labels_permitted_before_label_free_prediction_freeze": False,
        "eligible_for_model_or_scaler_fit": False, "eligible_for_final_blind_claim": True, "source_candidacy": "PROHIBITED_BLIND_TARGET_ONLY",
        "official_landing_url": "https://zenodo.org/records/545928", "persistent_id": "10.5281/zenodo.545928",
        "repository_version": "Zenodo version v1", "license": "CC BY-NC 4.0",
        "expected_images_or_scans": 200, "expected_patients": 78, "patient_grouping_status": "PATIENT_AND_LESION_GROUPS_EXPECTED_IN_RELEASE_VERIFY_IN_QUARANTINE",
        "positive_mapping": "sealed released pathology=malignant", "negative_mapping": "sealed released pathology=benign",
        "exclusions": "unknown|missing", "label_storage_coupling": "RAW_RF_AND_LABEL_METADATA_BUNDLED_IN_MAT_CONTAINER",
        "acquisition_gate": "POST_MODEL_FREEZE_PRESEALED_RF_RENDERER_PLUS_OPAQUE_LABEL_QUARANTINE_OR_MARK_UNAVAILABLE",
        "provenance_overlap_gate": "REQUIRE_CLEARANCE_AGAINST_BREAST_LESIONS_USG_BEFORE_ANY_BLIND_EDGE", "source_evidence_date": SOURCE_DATE,
    },
    {
        "dataset_id": "DERM7PT_2019", "modality": "dermoscopy", "task": "melanoma_vs_melanocytic_nevus",
        "role_frozen_before_label_access": BLIND, "acquisition_priority": 103,
        "analysis_use": "LOCKED_TARGET_ONLY_NEVER_SOURCE", "labels_permitted_before_label_free_prediction_freeze": False,
        "eligible_for_model_or_scaler_fit": False, "eligible_for_final_blind_claim": True, "source_candidacy": "PROHIBITED_BLIND_TARGET_ONLY",
        "official_landing_url": "https://derm.cs.sfu.ca/", "persistent_id": "10.1109/JBHI.2018.2824327",
        "repository_version": "authors' released Seven-Point Checklist dataset", "license": "CC BY-NC-SA 4.0",
        "expected_images_or_scans": 1011, "expected_patients": -1, "patient_grouping_status": "LESION_LEVEL_IDENTIFIERS_EXPECTED_VERIFY_WITHOUT_OUTCOME_USE",
        "positive_mapping": "sealed released diagnosis=melanoma", "negative_mapping": "sealed released diagnosis=melanocytic nevus",
        "exclusions": "non-melanoma and non-nevus diagnoses|clinical photographs|unknown|missing",
        "label_storage_coupling": "METADATA_EXPECTED_SEPARATE_FROM_IMAGES",
        "acquisition_gate": "POST_MODEL_FREEZE_DERMOSCOPY_ONLY_IMAGE_ACQUISITION_WITH_METADATA_QUARANTINE_OR_MARK_UNAVAILABLE",
        "provenance_overlap_gate": "REQUIRE_HASH_CLEARANCE_AGAINST_UDA1_MSK1_HAM10000", "source_evidence_date": SOURCE_DATE,
    },
]

role_registry = pd.DataFrame(dataset_rows).sort_values(["role_frozen_before_label_access", "acquisition_priority"]).reset_index(drop=True)
write_immutable_csv(ROLE_REGISTRY_PATH, role_registry)
ROLE_REGISTRY_HASH = sha256_file(ROLE_REGISTRY_PATH)

protocol_spec = {
    "stage": "Stage10",
    "scope": "PREASSIGNED_DEVELOPMENT_AND_LOCKED_BLIND_ROLE_REGISTRY_ONLY",
    "parent_stage9_final_record_sha256": EXPECTED_STAGE9_FINAL_HASH,
    "parent_stage9_protocol_seal_sha256": EXPECTED_STAGE9_PROTOCOL_SEAL,
    "model_specification_sha256": EXPECTED_MODEL_SPEC_HASH,
    "abstain_policy_sha256": EXPECTED_ABSTAIN_POLICY_HASH,
    "future_blind_protocol_sha256": EXPECTED_FUTURE_BLIND_PROTOCOL_HASH,
    "endpoint_specification_sha256": ENDPOINT_SPEC_HASH,
    "role_registry_sha256": ROLE_REGISTRY_HASH,
    "development_datasets": int(role_registry["role_frozen_before_label_access"].eq(DEVELOPMENT).sum()),
    "locked_blind_datasets": int(role_registry["role_frozen_before_label_access"].eq(BLIND).sum()),
    "roles_mutually_exclusive_and_immutable": True,
    "substitution_after_access_or_outcome_inspection_authorised": False,
    "blind_to_development_reassignment_authorised": False,
    "external_requests_in_stage10": False,
    "image_or_label_download_in_stage10": False,
    "new_outcomes_computed_in_stage10": False,
    "ddo2_fit_in_stage10": False,
    "blind_validation_in_stage10": False,
}

if PROTOCOL_SEAL_PATH.is_file():
    stage10_seal = json.loads(PROTOCOL_SEAL_PATH.read_text(encoding="utf-8"))
    seal_claim = stage10_seal["seal_sha256"]
    without_claim = dict(stage10_seal)
    without_claim.pop("seal_sha256")
    assert sha256_json(without_claim) == seal_claim
    assert stage10_seal["protocol_spec"] == protocol_spec
    assert stage10_seal["notebook_source_sha256"] == notebook_source_hash
    assert stage10_seal["parent_input_commitment_sha256"] == sha256_file(INPUT_COMMITMENT_PATH)
else:
    stage10_seal = {
        "stage": "Stage10",
        "decision": "PREASSIGNED_DATASET_ROLES_AND_ENDPOINTS_SEALED_BEFORE_NEW_DATA_ACCESS",
        "notebook_source_sha256": notebook_source_hash,
        "parent_input_commitment_sha256": sha256_file(INPUT_COMMITMENT_PATH),
        "protocol_spec": protocol_spec,
        "sealed_utc": utc_now(),
    }
    stage10_seal["seal_sha256"] = sha256_json(stage10_seal)
    seal_claim = stage10_seal["seal_sha256"]
    write_immutable_json(PROTOCOL_SEAL_PATH, stage10_seal)

runtime_state.update({"role_registry_sealed": True, "stage10_protocol_seal_sha256": seal_claim, "last_updated_utc": utc_now()})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Development / locked-blind datasets:", protocol_spec["development_datasets"], "/", protocol_spec["locked_blind_datasets"])
print("Endpoint specification hash:", ENDPOINT_SPEC_HASH)
print("Role registry hash:", ROLE_REGISTRY_HASH)
print("Stage 10 protocol seal:", seal_claim)


Development / locked-blind datasets: 5 / 3
Endpoint specification hash: 7eb31a4630c68488dac71d1602e4faf2dca348541d1a9ee62103524e8b83997c
Role registry hash: cc0b56178eecee123d8c6fa99824018b18aae79679c6620edc7e5ebcc9018bc3
Stage 10 protocol seal: 305c715cd95c72b7e54cfd241891080ddd6838e4428c6bd5acdf7b4264c96f31


In [10]:
# @title 10-2. Audit role exclusivity, licences, task mappings, and provenance hazards
stage9_role_template = pd.read_csv(STAGE9_ROLE_TEMPLATE_PATH)
assert set(stage9_role_template["role_frozen_before_label_access"]) == {DEVELOPMENT, BLIND}

audit_rows = []
def add_audit(check, passed, observed, expected, consequence="STOP_STAGE10_PROMOTION"):
    audit_rows.append({"check": check, "passed": bool(passed), "observed": str(observed), "expected": str(expected), "consequence_if_failed": consequence})

add_audit("unique_dataset_id", role_registry["dataset_id"].is_unique, role_registry["dataset_id"].nunique(), len(role_registry))
add_audit("allowed_roles_only", set(role_registry["role_frozen_before_label_access"]) == {DEVELOPMENT, BLIND}, sorted(role_registry["role_frozen_before_label_access"].unique()), [DEVELOPMENT, BLIND])
add_audit("unique_persistent_id", role_registry["persistent_id"].is_unique, role_registry["persistent_id"].nunique(), len(role_registry))
add_audit("unique_official_landing_url", role_registry["official_landing_url"].is_unique, role_registry["official_landing_url"].nunique(), len(role_registry))
add_audit("all_tasks_have_frozen_endpoint", set(role_registry["modality"]).issubset(ENDPOINT_SPECS), sorted(role_registry["modality"].unique()), sorted(ENDPOINT_SPECS))
add_audit("licence_declared_for_every_dataset", role_registry["license"].str.len().gt(0).all(), int(role_registry["license"].str.len().gt(0).sum()), len(role_registry))
add_audit("source_evidence_date_frozen", role_registry["source_evidence_date"].eq(SOURCE_DATE).all(), sorted(role_registry["source_evidence_date"].unique()), SOURCE_DATE)

dev = role_registry[role_registry["role_frozen_before_label_access"].eq(DEVELOPMENT)].copy()
blind = role_registry[role_registry["role_frozen_before_label_access"].eq(BLIND)].copy()
add_audit("development_labels_permitted", dev["labels_permitted_before_label_free_prediction_freeze"].all(), int(dev["labels_permitted_before_label_free_prediction_freeze"].sum()), len(dev))
add_audit("development_fit_eligible", dev["eligible_for_model_or_scaler_fit"].all(), int(dev["eligible_for_model_or_scaler_fit"].sum()), len(dev))
add_audit("development_not_blind_claim", (~dev["eligible_for_final_blind_claim"]).all(), int(dev["eligible_for_final_blind_claim"].sum()), 0)
add_audit("blind_labels_prohibited", (~blind["labels_permitted_before_label_free_prediction_freeze"]).all(), int(blind["labels_permitted_before_label_free_prediction_freeze"].sum()), 0)
add_audit("blind_never_fit_eligible", (~blind["eligible_for_model_or_scaler_fit"]).all(), int(blind["eligible_for_model_or_scaler_fit"].sum()), 0)
add_audit("blind_claim_eligible", blind["eligible_for_final_blind_claim"].all(), int(blind["eligible_for_final_blind_claim"].sum()), len(blind))
add_audit("blind_target_only", blind["analysis_use"].eq("LOCKED_TARGET_ONLY_NEVER_SOURCE").all(), sorted(blind["analysis_use"].unique()), "LOCKED_TARGET_ONLY_NEVER_SOURCE")
add_audit("minimum_two_complete_blind_datasets", len(blind) >= 2, len(blind), ">=2")
add_audit("no_data_acquired_by_stage10", True, 0, 0)
add_audit("no_external_url_requests_by_notebook", True, 0, 0)

role_exclusivity_audit = pd.DataFrame(audit_rows)
write_immutable_csv(AUDIT_ROOT / "Stage10_Role_Exclusivity_And_Protocol_Audit_v0.1.csv", role_exclusivity_audit)
if not role_exclusivity_audit["passed"].all():
    print(role_exclusivity_audit.loc[~role_exclusivity_audit["passed"]].to_string(index=False))
    raise AssertionError("Stage 10 role or endpoint audit failed.")

rights_audit = role_registry[[
    "dataset_id", "role_frozen_before_label_access", "official_landing_url", "persistent_id", "repository_version",
    "license", "source_evidence_date", "acquisition_gate", "label_storage_coupling",
]].copy()
rights_audit["stage10_network_check"] = "NOT_PERFORMED_STATIC_SOURCE_LEDGER_ONLY"
rights_audit["stage10_data_status"] = "NOT_ACQUIRED"
rights_audit["licence_consequence"] = "IF_TERMS_DIFFER_AT_ACQUISITION_MARK_UNAVAILABLE_DO_NOT_SUBSTITUTE"
write_immutable_csv(AUDIT_ROOT / "Stage10_Official_Source_Rights_And_Access_Ledger_v0.1.csv", rights_audit)

label_mapping_rows = []
for modality, spec in ENDPOINT_SPECS.items():
    label_mapping_rows.append({
        "modality": modality, "task": spec["task"], "positive_definition": spec["positive_class"],
        "negative_definition": spec["negative_class"], "excluded": "|".join(spec["excluded"]),
        "forbidden_surrogate": spec["forbidden_surrogate"], "grouping_unit": spec["grouping_unit"],
    })
write_immutable_csv(AUDIT_ROOT / "Stage10_Frozen_Label_Mapping_And_Grouping_Audit_v0.1.csv", pd.DataFrame(label_mapping_rows))

overlap_rows = []
dataset_ids = role_registry["dataset_id"].tolist()
for left_index, left in enumerate(dataset_ids):
    for right in dataset_ids[left_index + 1:]:
        pair = {left, right}
        if pair == {"BREAST_LESIONS_USG_2024", "OASBUD_2017"}:
            status = "POTENTIAL_COMMON_INSTITUTION_REQUIRES_PROVENANCE_CLEARANCE"
            consequence = "OASBUD_BLIND_EDGES_ABSTAIN_IF_INDEPENDENCE_NOT_PROVEN"
        elif "RODRIGUES_BUI_2017" in pair:
            status = "REQUIRE_IMAGE_HASH_AND_DERIVED_COPY_CLEARANCE"
            consequence = "RETIRE_DUPLICATE_ROWS_OR_DATASET_BEFORE_EDGE_AUTHORIZATION"
        else:
            status = "NO_KNOWN_OVERLAP_NOT_PROVEN"
            consequence = "REQUIRE_ARCHIVE_HASH_AND_PROVENANCE_AUDIT_AT_ACQUISITION"
        overlap_rows.append({"dataset_a": left, "dataset_b": right, "pre_access_status": status, "required_consequence": consequence})
overlap_audit = pd.DataFrame(overlap_rows)
write_immutable_csv(AUDIT_ROOT / "Stage10_Frozen_Dataset_Overlap_Gates_v0.1.csv", overlap_audit)

print("All role, task, label-mapping, and licence-ledger checks passed.")
print("Potential overlap pairs requiring special clearance:")
print(overlap_audit.loc[overlap_audit["pre_access_status"].ne("NO_KNOWN_OVERLAP_NOT_PROVEN")].to_string(index=False))


All role, task, label-mapping, and licence-ledger checks passed.
Potential overlap pairs requiring special clearance:
              dataset_a          dataset_b                                          pre_access_status                                       required_consequence
           BUS_BRA_2024 RODRIGUES_BUI_2017              REQUIRE_IMAGE_HASH_AND_DERIVED_COPY_CLEARANCE RETIRE_DUPLICATE_ROWS_OR_DATASET_BEFORE_EDGE_AUTHORIZATION
       BUSI_WHU_2025_V3 RODRIGUES_BUI_2017              REQUIRE_IMAGE_HASH_AND_DERIVED_COPY_CLEARANCE RETIRE_DUPLICATE_ROWS_OR_DATASET_BEFORE_EDGE_AUTHORIZATION
BREAST_LESIONS_USG_2024 RODRIGUES_BUI_2017              REQUIRE_IMAGE_HASH_AND_DERIVED_COPY_CLEARANCE RETIRE_DUPLICATE_ROWS_OR_DATASET_BEFORE_EDGE_AUTHORIZATION
BREAST_LESIONS_USG_2024        OASBUD_2017 POTENTIAL_COMMON_INSTITUTION_REQUIRES_PROVENANCE_CLEARANCE      OASBUD_BLIND_EDGES_ABSTAIN_IF_INDEPENDENCE_NOT_PROVEN
       BUS_UCLM_2025_V3 RODRIGUES_BUI_2017              REQUIRE_IMAGE_HASH_AN

In [11]:
# @title 10-3. Freeze development candidate edges, capacity logic, and deterministic stop rules
from itertools import permutations

development_edge_rows = []
for source, target in permutations(dev["dataset_id"].tolist(), 2):
    source_row = dev.loc[dev["dataset_id"].eq(source)].iloc[0]
    target_row = dev.loc[dev["dataset_id"].eq(target)].iloc[0]
    assert source_row["modality"] == target_row["modality"] == "breast_ultrasound"
    assert source_row["task"] == target_row["task"] == "breast_lesion_malignant_vs_benign"
    development_edge_rows.append({
        "edge_id": f"DEV::{source}__TO__{target}", "modality": "breast_ultrasound",
        "task": "breast_lesion_malignant_vs_benign", "source": source, "target": target,
        "edge_role": DEVELOPMENT, "source_gate_status": "PENDING_PATIENT_AND_SOURCE_RECOVERABILITY_GATES",
        "target_labels_allowed": True, "eligible_for_fit_if_source_gate_passes": True,
        "outcome_inspected_in_stage10": False, "activation_status": "PENDING_FIXED_PRIORITY_ACQUISITION",
    })
development_edges = pd.DataFrame(development_edge_rows).sort_values(["source", "target"]).reset_index(drop=True)
assert development_edges["edge_id"].is_unique and len(development_edges) == len(dev) * (len(dev) - 1)
write_immutable_csv(DEVELOPMENT_ROOT / "Stage10_Frozen_Development_Candidate_Edge_Registry_v0.1.csv", development_edges)

stage9_requirements = pd.read_csv(STAGE9_REQUIREMENTS_PATH)
stage9_shortfall = pd.read_csv(STAGE9_AXIS_SHORTFALL_PATH)
CURRENT_EDGES = int(stage9_final["eligible_edges"])
CURRENT_MODALITIES = int(stage9_final["modalities"])
CURRENT_DATASETS = int(stage9_final["unique_datasets"])
FIT_GATES = stage9_model["fit_gates"]

additional_edges_needed = max(0, int(FIT_GATES["minimum_total_eligible_edges"]) - CURRENT_EDGES)
outgoing_edges_per_recoverable_source = len(dev) - 1
minimum_recoverable_sources_needed = math.ceil(additional_edges_needed / outgoing_edges_per_recoverable_source)
conservative_realised_new_edges = minimum_recoverable_sources_needed * outgoing_edges_per_recoverable_source

capacity_rows = [
    {"gate": "total_eligible_edges", "current": CURRENT_EDGES, "projected_if_minimum_source_count_passes": CURRENT_EDGES + conservative_realised_new_edges, "required": FIT_GATES["minimum_total_eligible_edges"], "projection_is_outcome_free": True},
    {"gate": "modalities", "current": CURRENT_MODALITIES, "projected_if_minimum_source_count_passes": CURRENT_MODALITIES + 1, "required": FIT_GATES["minimum_modalities"], "projection_is_outcome_free": True},
    {"gate": "unique_datasets", "current": CURRENT_DATASETS, "projected_if_minimum_source_count_passes": CURRENT_DATASETS + len(dev), "required": FIT_GATES["minimum_unique_datasets"], "projection_is_outcome_free": True},
    {"gate": "locked_blind_datasets", "current": 0, "projected_if_minimum_source_count_passes": len(blind), "required": 2, "projection_is_outcome_free": True},
]
capacity_plan = pd.DataFrame(capacity_rows)
write_immutable_csv(DEVELOPMENT_ROOT / "Stage10_OutcomeFree_Capacity_Projection_v0.1.csv", capacity_plan)

acquisition_sequence = dev[["acquisition_priority", "dataset_id", "official_landing_url", "persistent_id", "acquisition_gate", "source_candidacy"]].copy()
acquisition_sequence["activation_rule"] = "ACTIVATE_IN_ASCENDING_PRIORITY_WITHOUT_USING_TRANSFER_OUTCOMES"
acquisition_sequence["failure_rule"] = "MARK_UNAVAILABLE_IN_FROZEN_ROLE_DO_NOT_SUBSTITUTE"
write_immutable_csv(DEVELOPMENT_ROOT / "Stage10_Frozen_Development_Acquisition_Sequence_v0.1.csv", acquisition_sequence)

development_stop_rules = {
    "candidate_development_datasets": dev["dataset_id"].tolist(),
    "maximum_candidate_directed_edges": int(len(development_edges)),
    "current_eligible_edges": CURRENT_EDGES,
    "additional_eligible_edges_needed_for_global_count_gate": int(additional_edges_needed),
    "outgoing_edges_per_recoverable_source_if_all_five_targets_acquired": int(outgoing_edges_per_recoverable_source),
    "minimum_recoverable_sources_needed_for_global_edge_count_gate": int(minimum_recoverable_sources_needed),
    "conservative_total_if_that_many_sources_pass": int(CURRENT_EDGES + conservative_realised_new_edges),
    "source_failure_rule": "retain the failed source-gate record and retire its outgoing edges; do not tune or replace it within the frozen protocol",
    "activation_rule": "use fixed acquisition priority; do not select a dataset because its transfer result looks favourable",
    "coefficient_fit_rule": "fit remains prohibited until every original Stage 9 global, grouped-fold, axis-class, and modality-class gate passes on realised eligible development edges",
    "axis_balance_warning": stage9_shortfall.to_dict("records"),
    "no_guarantee": "capacity arithmetic does not predict calibration or operating-point class balance",
}
development_stop_rules["rules_sha256"] = sha256_json(development_stop_rules)
write_immutable_json(DEVELOPMENT_ROOT / "Stage10_Frozen_Development_Stop_And_Fit_Gate_Rules_v0.1.json", development_stop_rules)

print("Development datasets / candidate directed edges:", len(dev), "/", len(development_edges))
print("Additional eligible edges needed:", additional_edges_needed)
print("Minimum recoverable ultrasound sources needed for edge-count gate:", minimum_recoverable_sources_needed, "of", len(dev))
print("Conservative projected total if that source count passes:", CURRENT_EDGES + conservative_realised_new_edges)
print("Axis-balance gates remain outcome-dependent and are not declared passed.")


Development datasets / candidate directed edges: 5 / 20
Additional eligible edges needed: 9
Minimum recoverable ultrasound sources needed for edge-count gate: 3 of 5
Conservative projected total if that source count passes: 33
Axis-balance gates remain outcome-dependent and are not declared passed.


In [12]:
# @title 10-4. Freeze locked-blind target edges and the label-quarantine contract
blind_edge_rows = []
ultrasound_blind_targets = blind.loc[blind["modality"].eq("breast_ultrasound"), "dataset_id"].tolist()
for source in dev["dataset_id"].tolist():
    for target in ultrasound_blind_targets:
        blind_edge_rows.append({
            "edge_id": f"BLIND::{source}__TO__{target}", "modality": "breast_ultrasound",
            "task": "breast_lesion_malignant_vs_benign", "source": source, "target": target,
            "edge_role": BLIND, "source_gate_status": "MUST_PASS_IN_DEVELOPMENT_BEFORE_BLIND_IMAGE_ACCESS",
            "target_role": "LOCKED_TARGET_ONLY_NEVER_SOURCE", "target_labels_accessed": False,
            "target_eligible_for_fit": False, "prediction_status": "BLOCKED_UNTIL_FINAL_MODEL_STATE_HASH_FROZEN",
            "unseal_count_allowed": 1,
        })

for source in ["UDA-1", "MSK-1", "HAM10000"]:
    blind_edge_rows.append({
        "edge_id": f"BLIND::{source}__TO__DERM7PT_2019", "modality": "dermoscopy",
        "task": "melanoma_vs_melanocytic_nevus", "source": source, "target": "DERM7PT_2019",
        "edge_role": BLIND, "source_gate_status": "REUSE_ONLY_THE_EXISTING_FROZEN_RECOVERABLE_SOURCE_AXIS",
        "target_role": "LOCKED_TARGET_ONLY_NEVER_SOURCE", "target_labels_accessed": False,
        "target_eligible_for_fit": False, "prediction_status": "BLOCKED_UNTIL_FINAL_MODEL_STATE_HASH_FROZEN",
        "unseal_count_allowed": 1,
    })

blind_edges = pd.DataFrame(blind_edge_rows).sort_values(["modality", "source", "target"]).reset_index(drop=True)
assert blind_edges["edge_id"].is_unique
assert not set(blind_edges["target"]) & set(blind_edges["source"])
assert set(blind_edges["target"]) == set(blind["dataset_id"])
assert (~blind_edges["target_labels_accessed"]).all() and (~blind_edges["target_eligible_for_fit"]).all()
write_immutable_csv(BLIND_ROOT / "Stage10_Frozen_Locked_Blind_Target_Edge_Registry_v0.1.csv", blind_edges)

blind_reserve_audit = pd.DataFrame([
    {"gate": "complete_locked_blind_datasets", "observed": len(blind), "required": 2, "passed": len(blind) >= 2},
    {"gate": "locked_blind_modalities", "observed": blind["modality"].nunique(), "required": 1, "passed": blind["modality"].nunique() >= 1},
    {"gate": "blind_targets_used_in_fit", "observed": int(blind["eligible_for_model_or_scaler_fit"].sum()), "required": 0, "passed": not blind["eligible_for_model_or_scaler_fit"].any()},
    {"gate": "blind_targets_used_as_sources", "observed": int(blind["dataset_id"].isin(blind_edges["source"]).sum()), "required": 0, "passed": not blind["dataset_id"].isin(blind_edges["source"]).any()},
    {"gate": "blind_labels_accessed_in_stage10", "observed": 0, "required": 0, "passed": True},
])
assert blind_reserve_audit["passed"].all()
write_immutable_csv(BLIND_ROOT / "Stage10_Locked_Blind_Reserve_Audit_v0.1.csv", blind_reserve_audit)

quarantine_contract = {
    "name": "Stage10_Locked_Blind_Label_Quarantine_Contract_v0.1",
    "parent_future_blind_protocol_sha256": EXPECTED_FUTURE_BLIND_PROTOCOL_HASH,
    "locked_targets": blind["dataset_id"].tolist(),
    "blind_edge_count": int(len(blind_edges)),
    "precondition_for_any_blind_image_acquisition": "a converged final DDO2 fit, immutable transforms, posterior draws, software environment, and final model-state hash must already exist",
    "precondition_for_prediction": "source axis and target image manifest hashes frozen; all label-free components computed without diagnostic labels",
    "bundled_label_rule": "a separate controlled extractor may create opaque image IDs without printing, joining, or exposing labels to analysis; if clean quarantine cannot be demonstrated, mark the dataset UNAVAILABLE_LOCKED",
    "path_and_filename_rule": "class-bearing paths and filenames may not enter embedding, prediction, logs, manifests, or analyst-visible output",
    "metadata_rule": "diagnostic metadata remain excluded until all three-axis predictions and ABSTAIN decisions are immutable",
    "one_time_unseal": "retrieve or decrypt each locked outcome table once after prediction freeze; evaluate and publish every authorised edge including failures and abstentions",
    "unavailable_rule": "an unavailable blind dataset remains locked and unavailable; it is never moved into development and is not replaced without a prospective protocol amendment",
    "overlap_rule": "any unresolved patient, image-hash, or provenance overlap forces ABSTAIN for every affected blind edge",
    "blind_labels_accessed_in_stage10": False,
    "blind_images_accessed_in_stage10": False,
}
quarantine_contract["contract_sha256"] = sha256_json(quarantine_contract)
write_immutable_json(BLIND_ROOT / "Stage10_Frozen_Locked_Blind_Label_Quarantine_Contract_v0.1.json", quarantine_contract)

print("Locked-blind datasets / modalities / candidate edges:", len(blind), "/", blind["modality"].nunique(), "/", len(blind_edges))
print("Blind images / labels accessed:", False, "/", False)
print("Quarantine contract hash:", quarantine_contract["contract_sha256"])


Locked-blind datasets / modalities / candidate edges: 3 / 2 / 13
Blind images / labels accessed: False / False
Quarantine contract hash: d72d7b859cc1ddf07f94154c5ecdb540976a79bae2a348dbb7c7d641fd4567b9


In [13]:
# @title 10-5. Create the development-only Stage 11 handoff and amendment boundary
stage11_manifest = dev[[
    "acquisition_priority", "dataset_id", "modality", "task", "official_landing_url", "persistent_id",
    "repository_version", "license", "expected_images_or_scans", "expected_patients", "patient_grouping_status",
    "positive_mapping", "negative_mapping", "exclusions", "acquisition_gate", "source_candidacy",
]].copy()
stage11_manifest["authorised_next_stage"] = "STAGE11_DEVELOPMENT_ACQUISITION_AND_SOURCE_GATE_ONLY"
stage11_manifest["locked_blind_asset"] = False
stage11_manifest["outcome_based_selection_allowed"] = False
write_immutable_csv(DEVELOPMENT_ROOT / "Stage10_Stage11_Development_Only_Acquisition_Manifest_v0.1.csv", stage11_manifest)

handoff_rules = {
    "next_stage": "Stage11_Development_Extension_Acquisition_And_Source_Recoverability_Gates",
    "authorised_assets": dev["dataset_id"].tolist(),
    "prohibited_assets": blind["dataset_id"].tolist(),
    "stage11_scope": [
        "verify official versions, licences, checksums, schemas, patient groups, duplicates, and provenance",
        "acquire development assets only in frozen priority order",
        "freeze deterministic ultrasound rendering and image normalisation without using transfer outcomes",
        "perform patient-grouped source recoverability gates and retain all failures",
        "do not fit final DDO2 and do not touch locked-blind images or labels",
    ],
    "dataset_unavailable_rule": "record UNAVAILABLE under the original role; no silent mirror, no ad hoc substitute, no role transfer",
    "amendment_rule": "any replacement requires a new prospective Stage10 amendment sealed before replacement data or labels are accessed",
    "promotion_rule": "a later development-edge stage may proceed only after Stage11 source axes, manifests, transforms, and hashes are frozen",
    "coefficient_fit_rule": "even after development edges are evaluated, fitting is authorised only if every original Stage9 gate passes",
    "locked_blind_rule": "blind acquisition remains prohibited until the final fitted model-state hash exists",
}
handoff_rules["handoff_sha256"] = sha256_json(handoff_rules)
write_immutable_json(DEVELOPMENT_ROOT / "Stage10_Stage11_Handoff_And_Amendment_Rules_v0.1.json", handoff_rules)

role_summary = (
    role_registry.groupby(["role_frozen_before_label_access", "modality"], as_index=False)
    .agg(datasets=("dataset_id", "nunique"), expected_images_or_scans=("expected_images_or_scans", "sum"))
)
write_immutable_csv(RESULT_ROOT / "Stage10_Role_And_Modality_Summary_v0.1.csv", role_summary)

runtime_state.update({
    "development_handoff_frozen": True, "development_datasets": int(len(dev)),
    "locked_blind_datasets": int(len(blind)), "candidate_development_edges": int(len(development_edges)),
    "candidate_blind_edges": int(len(blind_edges)), "external_requests_made": False,
    "image_archives_downloaded": False, "future_labels_accessed": False, "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Stage 11 development-only handoff hash:", handoff_rules["handoff_sha256"])
print("Authorised next-stage datasets:", dev["dataset_id"].tolist())
print("Locked and prohibited in Stage 11:", blind["dataset_id"].tolist())


Stage 11 development-only handoff hash: 9424f1533391db7b56d93a61af051865ef2176257d1b031a2b2b268755342f8b
Authorised next-stage datasets: ['BUS_BRA_2024', 'BUSI_WHU_2025_V3', 'BREAST_LESIONS_USG_2024', 'BUS_UCLM_2025_V3', 'RODRIGUES_BUI_2017']
Locked and prohibited in Stage 11: ['BUSI_CAIRO_2019', 'OASBUD_2017', 'DERM7PT_2019']


In [14]:
# @title 10-6. Seal the decision record, integrity manifest, and handoff report
import matplotlib.pyplot as plt

if not FIGURE_PATH.is_file():
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
    role_counts = role_registry["role_frozen_before_label_access"].value_counts().reindex([DEVELOPMENT, BLIND])
    axes[0].bar(["Development", "Locked blind"], role_counts.values, color=["#457b9d", "#e76f51"])
    axes[0].set_ylabel("Complete datasets")
    axes[0].set_title("Frozen acquisition roles")
    axes[0].grid(axis="y", alpha=0.2)

    labels = ["Current", "Min-source\nprojection", "Max candidate\nprojection"]
    values = [CURRENT_EDGES, CURRENT_EDGES + conservative_realised_new_edges, CURRENT_EDGES + len(development_edges)]
    colors = ["#6c757d", "#2a9d8f", "#457b9d"]
    axes[1].bar(labels, values, color=colors)
    axes[1].axhline(FIT_GATES["minimum_total_eligible_edges"], color="#d62828", linestyle="--", label="Stage 9 minimum = 30")
    axes[1].set_ylabel("Eligible directed edges")
    axes[1].set_title("Outcome-free edge-capacity arithmetic")
    axes[1].legend(frameon=False, fontsize=8)
    axes[1].grid(axis="y", alpha=0.2)
    fig.suptitle("Stage 10 development capacity and blind reserve", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)

decision = "SEAL_PREASSIGNED_DEVELOPMENT_EXTENSION_AND_LOCKED_BLIND_REGISTRY_AUTHORISE_DEVELOPMENT_ACQUISITION_ONLY"
next_step = "BUILD_STAGE11_DEVELOPMENT_EXTENSION_ACQUISITION_AND_SOURCE_RECOVERABILITY_GATES_WITHOUT_TOUCHING_LOCKED_BLIND_ASSETS"

report = f'''# Stage 10 — Preassigned Development Extension and Locked-Blind Dataset Registry Report v0.1

## Decision

`{decision}`

Stage 10 verified the immutable Stage 9 method, ABSTAIN, and future-blind specifications and froze **{len(dev)} development-extension datasets** and **{len(blind)} locked-blind datasets** before any new image or diagnostic-label access. No external dataset request, image download, label read, transfer evaluation, coefficient fit, or blind validation occurred.

## Development capacity

The new development modality is breast ultrasound with a pathology-compatible malignant-versus-benign lesion endpoint. Five preassigned domains create **{len(development_edges)} candidate directed edges**. The existing library contains **{CURRENT_EDGES} eligible edges** and needs **{additional_edges_needed}** more for the global count gate. With five development domains, each recoverable source can contribute four outgoing edges; therefore at least **{minimum_recoverable_sources_needed} of 5** sources must pass to reach a conservative total of **{CURRENT_EDGES + conservative_realised_new_edges}** edges.

This is capacity arithmetic, not an outcome claim. Calibration and operating-point outcomes currently need three additional pass edges each, and every grouped-fold and modality-class gate must still pass before coefficient fitting is authorised.

## Locked blind reserve

The permanently locked targets are `{', '.join(blind['dataset_id'])}` across breast ultrasound and dermoscopy. They create **{len(blind_edges)} preassigned target-only candidate edges**. They cannot be sources, cannot enter model or scaler fitting, and cannot be acquired during Stage 11. If bundled labels cannot be cleanly quarantined, or provenance independence cannot be proved, affected datasets/edges remain locked and are marked unavailable or ABSTAIN; they are not moved to development.

## Governance boundary

- Roles are mutually exclusive and immutable under this protocol.
- Dataset access or licence failure is recorded; no unofficial mirror or ad hoc replacement is authorised.
- Any replacement requires a prospective Stage 10 amendment before replacement data or labels are accessed.
- Stage 11 may acquire and source-gate development datasets only.
- Blind acquisition begins only after transforms, coefficients, posterior draws, software versions, and the final DDO2 model-state hash are frozen.

## Integrity identifiers

- Parent Stage 9 final record: `{EXPECTED_STAGE9_FINAL_HASH}`
- Parent Stage 9 protocol seal: `{EXPECTED_STAGE9_PROTOCOL_SEAL}`
- Frozen Stage 9 model specification: `{EXPECTED_MODEL_SPEC_HASH}`
- Stage 10 protocol seal: `{seal_claim}`
- Endpoint specification file: `{ENDPOINT_SPEC_HASH}`
- Dataset role registry: `{ROLE_REGISTRY_HASH}`
- Locked-blind quarantine contract: `{quarantine_contract['contract_sha256']}`
'''
write_immutable_text(REPORT_PATH, report)

output_files = [
    PROTOCOL_SEAL_PATH, INPUT_COMMITMENT_PATH, ENDPOINT_SPEC_PATH, ROLE_REGISTRY_PATH,
    AUDIT_ROOT / "Stage10_Role_Exclusivity_And_Protocol_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage10_Official_Source_Rights_And_Access_Ledger_v0.1.csv",
    AUDIT_ROOT / "Stage10_Frozen_Label_Mapping_And_Grouping_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage10_Frozen_Dataset_Overlap_Gates_v0.1.csv",
    DEVELOPMENT_ROOT / "Stage10_Frozen_Development_Candidate_Edge_Registry_v0.1.csv",
    DEVELOPMENT_ROOT / "Stage10_OutcomeFree_Capacity_Projection_v0.1.csv",
    DEVELOPMENT_ROOT / "Stage10_Frozen_Development_Acquisition_Sequence_v0.1.csv",
    DEVELOPMENT_ROOT / "Stage10_Frozen_Development_Stop_And_Fit_Gate_Rules_v0.1.json",
    DEVELOPMENT_ROOT / "Stage10_Stage11_Development_Only_Acquisition_Manifest_v0.1.csv",
    DEVELOPMENT_ROOT / "Stage10_Stage11_Handoff_And_Amendment_Rules_v0.1.json",
    BLIND_ROOT / "Stage10_Frozen_Locked_Blind_Target_Edge_Registry_v0.1.csv",
    BLIND_ROOT / "Stage10_Locked_Blind_Reserve_Audit_v0.1.csv",
    BLIND_ROOT / "Stage10_Frozen_Locked_Blind_Label_Quarantine_Contract_v0.1.json",
    RESULT_ROOT / "Stage10_Role_And_Modality_Summary_v0.1.csv", REPORT_PATH, FIGURE_PATH,
]
assert all(path.is_file() for path in output_files)
output_integrity = pd.DataFrame([
    {"relative_path": str(path.relative_to(STAGE10_ROOT)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in sorted(output_files, key=lambda item: str(item))
])
write_immutable_csv(OUTPUT_MANIFEST_PATH, output_integrity)

tracked_bytes = sum(path.stat().st_size for path in output_files + [OUTPUT_MANIFEST_PATH])
assert tracked_bytes <= MAXIMUM_NEW_STAGE10_BYTES

if FINAL_RECORD_PATH.is_file():
    final_record = verify_self_hashed_json(FINAL_RECORD_PATH, "final_record_sha256", json.loads(FINAL_RECORD_PATH.read_text(encoding="utf-8"))["final_record_sha256"])
    final_claim = final_record["final_record_sha256"]
    assert final_record["decision"] == decision
    assert final_record["stage10_protocol_seal_sha256"] == seal_claim
    assert final_record["role_registry_sha256"] == ROLE_REGISTRY_HASH
else:
    final_record = {
        "stage": "Stage10", "decision": decision,
        "scope": "ROLE_REGISTRY_AND_ACQUISITION_GOVERNANCE_ONLY_NOT_DATA_ACQUISITION_NOT_MODEL_FIT_NOT_BLIND_VALIDATION",
        "parent_stage9_final_record_sha256": EXPECTED_STAGE9_FINAL_HASH,
        "parent_stage9_protocol_seal_sha256": EXPECTED_STAGE9_PROTOCOL_SEAL,
        "model_specification_sha256": EXPECTED_MODEL_SPEC_HASH,
        "abstain_policy_sha256": EXPECTED_ABSTAIN_POLICY_HASH,
        "future_blind_protocol_sha256": EXPECTED_FUTURE_BLIND_PROTOCOL_HASH,
        "stage10_protocol_seal_sha256": seal_claim,
        "notebook_source_sha256": notebook_source_hash,
        "endpoint_specification_sha256": ENDPOINT_SPEC_HASH,
        "role_registry_sha256": ROLE_REGISTRY_HASH,
        "development_datasets": int(len(dev)), "locked_blind_datasets": int(len(blind)),
        "development_modalities_added": 1, "locked_blind_modalities": int(blind["modality"].nunique()),
        "candidate_development_edges": int(len(development_edges)), "candidate_locked_blind_edges": int(len(blind_edges)),
        "minimum_recoverable_development_sources_for_global_edge_count_gate": int(minimum_recoverable_sources_needed),
        "conservative_total_edges_if_minimum_source_count_passes": int(CURRENT_EDGES + conservative_realised_new_edges),
        "stage10_external_requests_made": False, "new_images_downloaded": False, "new_labels_accessed": False,
        "new_transfer_outcomes_computed": False, "ddo2_coefficients_fitted": False,
        "locked_blind_predictions_frozen": False, "blind_validation_performed": False,
        "quarantine_contract_sha256": quarantine_contract["contract_sha256"],
        "output_integrity_manifest_sha256": sha256_file(OUTPUT_MANIFEST_PATH),
        "tracked_stage10_output_bytes_excluding_runtime_and_final_record": int(tracked_bytes),
        "maximum_new_stage10_bytes": int(MAXIMUM_NEW_STAGE10_BYTES),
        "next_step": next_step, "completed_utc": utc_now(),
    }
    final_record["final_record_sha256"] = sha256_json(final_record)
    final_claim = final_record["final_record_sha256"]
    write_immutable_json(FINAL_RECORD_PATH, final_record)

runtime_state.update({
    "completed": True, "decision": decision, "final_record_sha256": final_claim,
    "external_requests_made": False, "image_archives_downloaded": False, "future_labels_accessed": False,
    "ddo2_coefficients_fitted": False, "blind_validation_performed": False, "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("\n================ STAGE 10 PREASSIGNED ROLE REGISTRY COMPLETE ================")
print("Development / locked-blind datasets:", len(dev), "/", len(blind))
print("Candidate development / locked-blind edges:", len(development_edges), "/", len(blind_edges))
print("New images / labels accessed:", False, "/", False)
print("DDO2 fitted / blind validation performed:", False, "/", False)
print("Decision:", decision)
print("Protocol seal:", seal_claim)
print("Role registry hash:", ROLE_REGISTRY_HASH)
print("Final record hash:", final_claim)
actual_stage10_bytes = sum(path.stat().st_size for path in STAGE10_ROOT.rglob("*") if path.is_file())
print("Actual Stage 10 bytes:", actual_stage10_bytes)
print("Next step:", next_step)



================ STAGE 10 PREASSIGNED ROLE REGISTRY COMPLETE ================
Development / locked-blind datasets: 5 / 3
Candidate development / locked-blind edges: 20 / 13
New images / labels accessed: False / False
DDO2 fitted / blind validation performed: False / False
Decision: SEAL_PREASSIGNED_DEVELOPMENT_EXTENSION_AND_LOCKED_BLIND_REGISTRY_AUTHORISE_DEVELOPMENT_ACQUISITION_ONLY
Protocol seal: 305c715cd95c72b7e54cfd241891080ddd6838e4428c6bd5acdf7b4264c96f31
Role registry hash: cc0b56178eecee123d8c6fa99824018b18aae79679c6620edc7e5ebcc9018bc3
Final record hash: 6438434cb41607ad97b1b4a2fab07b143969ce7f551b87b5c2a23afbe67ccccf
Actual Stage 10 bytes: 131890
Next step: BUILD_STAGE11_DEVELOPMENT_EXTENSION_ACQUISITION_AND_SOURCE_RECOVERABILITY_GATES_WITHOUT_TOUCHING_LOCKED_BLIND_ASSETS
